# Diferencias en el consumo de potencia entre instrucciones (PRINCIPAL)

**RESUMEN:** *Ahora que te has familiarizado con la plataforma ChipWhisperer, la utilizaremos para aprender más sobre cómo varía la potencia consumida por un microcontrolador según las operaciones que realiza y las instrucciones que ejecuta.*

**OBJETIVOS DE APRENDIZAJE:**

* Capturar una traza de potencia con ChipWhisperer
* Establecer conexiones entre varias instrucciones simples y las trazas de potencia

## Setup

Para configurar el hardware y software ejecutamos completo el notebook `2.Diferencias_consumo_potencia_(HARDWARE)`. 

A continuación se ejecuta una captura de prueba, asegúrate de obtener el mensaje `"✔️ OK to continue!"` una vez ejecutada; de lo contrario, ¡fallará más adelante!

In [ ]:
%run 2.Diferencias_consumo_potencia_(HARDWARE).ipynb

In [ ]:
wave = capture_trace()
print("✔️ OK to continue!")

## Traza "vacía"

Como viste en las prácticas anteriores, el firmware `simpleserial-base` devuelve como respuesta (*echo*) lo que le enviamos. Para esta práctica, solo queremos centrarnos en instrucciones simples, en lugar de la comunicación serie. Abre `simpleserial-base.c` en la carpeta `workspace/Firmwares/simpleserial-base-lab2`, dirígete a la función `get_pt()` y elimina la llamada a `simpleserial_put()`.

Una vez modificado el código, volvemos a compilar el firmware:

In [ ]:
%%bash -s "$PLATFORM"
cd Firmwares/simpleserial-base-lab2
make PLATFORM=$1 CRYPTO_TARGET=NONE -j

Lo cargamos en el objetivo:

In [ ]:
cw.program_target(scope, prog, "Firmwares/simpleserial-base-lab2/simpleserial-base-{}.hex".format(PLATFORM))

Capturamos una traza y la graficamos:

In [ ]:
wave = capture_trace()
cw.plot(wave)

No deberías ver gran actividad aquí antes de que el objetivo pase al estado de inactividad (*idle*).

## Instrucciones simples

Para empezar, insertemos algunas instrucciones simples en la función `get_pt()` para ver si son visibles en la traza de potencia. Buscamos principalmente una instrucción que se ejecute en un solo ciclo. Para un dispositivo ARM como el de nuestro obejtivo, intenta insertar el siguiente código, que ejecutará 20 multiplicaciones. Es importante declarar `A` como `volatile` en este punto para evitar que el compilador optimice y elimine todas estas instrucciones (las instrucciones deben insertarse después de activar el trigger):

```C
	volatile long int A = 0x2BAA;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;

	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
```

Ahora vuelve a compilar el firmware:

Súbelo al objetivo:

Y captura una nueva traza:

In [ ]:
wave2 = capture_trace()

cw.plot(wave2)

Dependiendo de tu objetivo, puede que no haya mucha diferencia aquí. Quizás prefieras graficar ambas trazas en el mismo gráfico:

In [ ]:
cw.plot(wave) * cw.plot(wave2)

## Bucle de instrucciones simples

Podrías pensar que es extraño que no hayamos usado simplemente un bucle en lugar de copiar lo mismo 20 veces. Corrige esto reemplazando estas instrucciones por un bucle. **Asegúrate de declarar la variable del bucle también como `volatile`.**


In [ ]:
wave3 = capture_trace()
cw.plot(wave3)

¿Te sorprende lo diferente que resulta el bucle en comparación con la simple repetición de operaciones? Si te manejas bien con ensamblador, puedes revisar el archivo `.lss` en el directorio `simpleserial-base-lab2`. De lo contrario, aquí tienes una descripción a nivel de C de cómo se ve realmente el bucle (según cómo organice el código tu compilador):

```C
i=0;
start:
    if (i > 20) {
        goto end;
    }    
    A *= 2;
    i += 1;
    goto start;
end:
    trigger_low();
    //...
```

Como puedes ver, el microcontrolador tiene que hacer mucho más que simplemente repetir nuestra instrucción. Por ello, es algo que un compilador a menudo "desenrollará" (básicamente, convertirlo de nuevo en nuestras multiplicaciones repetidas originales). En nuestro caso, dado que `i` se declaró como `volatile`, el compilador evitó aplicar esta optimización. Desenrollar bucles (loop unrolling) también incrementa el tamaño del código, por lo que el compilador también podría evitar desenrollar al optimizar para reducir el tamaño del binario.

## Instrucciones costosas

Hasta ahora, todas las instrucciones que hemos utilizado (aparte de los saltos o bifurcaciones en el bucle) se han ejecutado en un solo ciclo de reloj. En su lugar, probemos una instrucción que tarde varios ciclos de reloj en ejecutarse. Esto dependerá de la plataforma específica, pero para un dispositivo ARM, una división funcionará bien (no dividas por una potencia de 2 ya que el compilador la puede sustituir por un desplazamiento que se ejecutaría también en 1 ciclo).

Modifica el firmware utilizando divisiones, compílalo, súbelo al objetivo y captura una nueva traza:

In [ ]:
wave4 = capture_trace()
cw.plot(wave4)

Pintemos la anterior justo debajo:

In [ ]:
cw.plot(wave3)

Y ahora las dos juntas:

In [ ]:
cw.plot(wave4) * cw.plot(wave3)

Probablemente puedas apreciar un patrón similar al de la instrucción más rápida, pero cada iteración del bucle tarda más tiempo.

## Conclusiones y próximos pasos

A estas alturas ya deberías estar razonablemente convencido de que podemos comprender en parte lo que hace un microcontrolador observando su consumo de potencia. De aquí en adelante nos centraremos en mediciones más objetivas, pero lo que hicimos en esta práctica sigue siendo sumamente valioso. Las implementaciones de AES, por ejemplo, suelen presentar una forma muy característica. Puedes usar esto como ayuda para identificar lo que un dispositivo desconocido está haciendo en distintos momentos.

Por supuesto, hay mucho más que puedes hacer con esta práctica. Por ejemplo, ¿se ve una suma diferente a una operación XOR? La próxima vez, utilizaremos lo aprendido en esta práctica para vulnerar una comprobación simple de contraseña.

---
**Agradecimientos / Referencias:**  
Este material está basado y adaptado a partir del notebook original [*Lab 2_1A - Instruction Power Differences (MAIN)*](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/sca101/Lab%202_1A%20-%20Instruction%20Power%20Differences%20(MAIN).ipynb) del repositorio oficial de [NewAE Technology](https://github.com/newaetech/chipwhisperer-jupyter).